In [1]:
dir_path = '/mnt/sharedstorage/sabdelazim/Downloads'

In [2]:
%matplotlib inline

from pathlib import Path
import os
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm import tqdm
from megadetector.detection.run_detector_batch import load_and_run_detector_batch

In [3]:
def draw_md_boxes_on_image(img, detections, conf_thresh=0.2, animals_only=False):
    """
    Draw MegaDetector bounding boxes on an image.

    Categories:
    1 = animal
    2 = person
    3 = vehicle
    """
    img = img.copy()
    draw = ImageDraw.Draw(img)

    W, H = img.size

    for det in detections:
        conf = float(det.get("conf", 0))
        cat = str(det.get("category", ""))
        bbox = det.get("bbox")

        if bbox is None or conf < conf_thresh:
            continue

        if animals_only and cat != "1":
            continue

        x, y, w, h = bbox

        x1 = int(x * W)
        y1 = int(y * H)
        x2 = int((x + w) * W)
        y2 = int((y + h) * H)

        if cat == "1":
            color = "red"
        elif cat == "2":
            color = "blue"
        elif cat == "3":
            color = "yellow"
        else:
            color = "white"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        draw.text((x1 + 5, y1 - 10), f"{cat}:{conf:.2f}", fill=color)

    return img

def show_before_after(original, annotated, title=""):
    
    print(title)
    
    display(original)
    display(annotated)

In [4]:
dir_path = "/mnt/sharedstorage/sabdelazim/Downloads"

image_paths = sorted(
    list(Path(dir_path).glob("*.JPG")) +
    list(Path(dir_path).glob("*.jpg"))
)

print("Images found:", len(image_paths))

results = load_and_run_detector_batch(
    model_file="MDV5A",
    image_file_names=[str(p) for p in image_paths],
    confidence_threshold=0.2,
    quiet=True,
    batch_size=1
)

Images found: 8
Model v5a.0.1 already exists and is valid at /tmp/megadetector_models/md_v5a.0.1.pt
PyTorch reports 2 available CUDA devices
GPU available: True
PyTorch reports 2 available CUDA devices
GPU available: True


2026-03-10 22:59:47.826934: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-10 22:59:47.851888: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 22:59:47.851909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 22:59:47.852853: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-10 22:59:47.857401: I tensorflow/core/platform/cpu_feature_guar

Loading PT detector with compatibility mode classic
Loaded image size 1280 from model metadata
Using model stride: 64
PTDetector using device cuda:0


Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


Loaded model in 3.6 seconds


100%|█████████████████████████████████████████████| 8/8 [00:00<00:00, 15.90it/s]


In [11]:
# summary = []

# for r in tqdm(results):

#     fpath = r["file"]
#     detections = r.get("detections", [])

#     img = Image.open(fpath).convert("RGB")

#     annotated = draw_md_boxes_on_image(
#         img,
#         detections,
#         conf_thresh=0.2,
#         animals_only=False
#     )

#     show_before_after(
#         img,
#         annotated,
#         title=os.path.basename(fpath)
#     )

#     summary.append({
#         "filename": os.path.basename(fpath),
#         "detections": len(detections)
#     })

# summary_df = pd.DataFrame(summary)
# summary_df.head()

In [7]:
dir_path = Path("/mnt/sharedstorage/sabdelazim/images/all_species_images")

image_paths = sorted(
    list(dir_path.glob("*.JPG")) +
    list(dir_path.glob("*.jpg"))
)

print("Total images found:", len(image_paths))

# take first 2 images
test_images = image_paths[4:15]

test_images

Total images found: 420466


[PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010003_g08_waterbuck.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010004_g08_genet.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010005_g08_genet.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010005_m08_warthog.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010006_m08_warthog.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010023_m08_bushbuck.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010024_m08_bushbuck.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010025_m08_bushbuck.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010025_m08_warthog.jpg'),
 PosixPath('/mnt/sharedstorage/sabdelazim/images/all_species_images/01010026_m08_bushbuck.jpg'),
 PosixPath('/mnt/sharedstorage/sabdela

In [8]:
results = load_and_run_detector_batch(
    model_file="MDV5A",
    image_file_names=[str(p) for p in test_images],
    confidence_threshold=0.2,
    quiet=True,
    batch_size=1
)

Fusing layers... 
Fusing layers... 


Model v5a.0.1 already exists and is valid at /tmp/megadetector_models/md_v5a.0.1.pt
PyTorch reports 2 available CUDA devices
GPU available: True
PyTorch reports 2 available CUDA devices
GPU available: True
Bypassing imports for model type yolov5
Loading PT detector with compatibility mode classic
Loaded image size 1280 from model metadata
Using model stride: 64
PTDetector using device cuda:0


Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


Loaded model in 0.23 seconds


100%|███████████████████████████████████████████| 11/11 [00:00<00:00, 22.57it/s]


In [10]:
# summary = []

# for r in tqdm(results):

#     fpath = r["file"]
#     detections = r.get("detections", [])

#     img = Image.open(fpath).convert("RGB")

#     annotated = draw_md_boxes_on_image(
#         img,
#         detections,
#         conf_thresh=0.2,
#         animals_only=False
#     )

#     show_before_after(img, annotated)

#     summary.append({
#         "filename": Path(fpath).name,
#         "detections": len(detections)
#     })

# summary_df = pd.DataFrame(summary)
# summary_df